### **Tratamento dos microdados de Eficiência Acadêmica**

In [11]:
import pandas as pd

# Seleção das colunas de interesse para o DataFrame final
colunas_interesse = [
    'Código do Ciclo Matricula',
    'Instituição',               
    'Unidade de Ensino',         
    'Nome de Curso',             
    'Tipo de Curso',
    'Fator Esforço Curso',       
    'Carga Horaria',                       
    'Modalidade de Ensino'       
]

Na análise inicial da estrutura do dataframe, constatamos que a coluna `'tipo_oferta'` apresenta apenas valores registrados como `['Não se aplica']`, isto é, não temos registros do tipo de ingresso para esse conjunto de dados. Por esse motivo, a coluna em questão foi removida.

**Carga dos dados**

In [12]:
# Carregar o arquivo bruto em blocos
chunks = pd.read_csv('../data/raw/microdados_eficiencia_academica_2023.csv', sep=';', encoding='utf-8', usecols=colunas_interesse, chunksize=100000)

# Definir os tipos de graduação para filtrar os dados
tipos_graduacao = ['Licenciatura', 'Bacharelado', 'Tecnologia']

# Aplicar os filtros e concatenar os resultados em um DataFrame final
df_saida = pd.concat(
    [chunk[
        chunk['Instituição'].str.contains('BAHIA', na=False, case=False) &
        chunk['Tipo de Curso'].isin(tipos_graduacao)
    ] for chunk in chunks],

    ignore_index=True
)

# Remover coluna 'Instituição'
df_saida = df_saida.drop(columns=['Instituição'])

In [26]:
# Visualizar tabela resultante
df_saida.info()

<class 'pandas.DataFrame'>
RangeIndex: 2241 entries, 0 to 2240
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   carga_horaria           2241 non-null   int64  
 1   codigo_ciclo_matricula  2241 non-null   str    
 2   fator_esforco_curso     2241 non-null   float64
 3   modalidade_ensino       2241 non-null   str    
 4   nome_curso              2241 non-null   str    
 5   tipo_curso              2241 non-null   str    
 6   unidade_ensino          2241 non-null   str    
dtypes: float64(1), int64(1), str(5)
memory usage: 290.2 KB


In [18]:
df_saida.head()

,carga_horaria,codigo_ciclo_matricula,fator_esforco_curso,modalidade_ensino,nome_curso,tipo_curso,unidade_ensino
0,3630,2146517,"1,108",Educação Presencial,Engenharia Elétrica,Bacharelado,Campus Paulo Afonso
1,3630,2146517,"1,108",Educação Presencial,Engenharia Elétrica,Bacharelado,Campus Paulo Afonso
2,3630,2146517,"1,108",Educação Presencial,Engenharia Elétrica,Bacharelado,Campus Paulo Afonso
3,3630,2146517,"1,108",Educação Presencial,Engenharia Elétrica,Bacharelado,Campus Paulo Afonso
4,3630,2146517,"1,108",Educação Presencial,Engenharia Elétrica,Bacharelado,Campus Paulo Afonso


**Tratamento dos dados**

In [15]:
# Renomear colunas
df_saida.columns = [
    'carga_horaria', 'codigo_ciclo_matricula',
    'fator_esforco_curso', 'modalidade_ensino',
    'nome_curso', 'tipo_curso', 'unidade_ensino'
]

In [22]:
# Converter o formato da coluna 'fator_esforco_curso' de string para float
df_saida['fator_esforco_curso'] = (
    df_saida['fator_esforco_curso']
    .str.replace(',', '.', regex=False)
    .astype('float64')
)

In [25]:
# Converter o formato da coluna 'codigo_ciclo_matricula' de inteiro para string
df_saida['codigo_ciclo_matricula'] = df_saida['codigo_ciclo_matricula'].astype(str)

**Persistência do DataFrame final tratado**

In [27]:
# Salvar o DataFrame resultante em arquivo Parquet

df_saida.to_parquet('../data/processed/microdados_eficiencia_academica_tratados.parquet', index=False)